In [ ]:
# Lecture 1 — The Big Data Problem
# Spark 101 · Module 1 · Lecture 1
#
# Import this file into a Databricks workspace as a notebook.
# Databricks recognizes `# COMMAND ----------` as a cell boundary on import.
# In VS Code / PyCharm this stays a plain Python file — same source, two homes.
#
# Demo: pandas hits the memory wall on a single machine (the Driver).
# This is the problem Spark exists to solve — run it BEFORE we ever touch Spark.

In [ ]:
import csv, os, random, time
import pandas as pd
import psutil

BASE = "/tmp/spark101_l01"
os.makedirs(BASE, exist_ok=True)
DATA = f"{BASE}/events_large.csv"
TARGET_ROWS = 20_000_000   # tune to the Driver's memory — bigger = closer to OOM


def rss_gb():
    # Resident memory of THIS process (the Driver), in GB.
    return psutil.Process(os.getpid()).memory_info().rss / 1024**3


print("Driver RSS at start:", round(rss_gb(), 2), "GB")

In [ ]:
# --- Generate a synthetic CSV, larger than we want to hold in memory --------
# Run this once; it takes a minute or two. Skips if the file already exists.
if not os.path.exists(DATA):
    print(f"Generating {TARGET_ROWS:,} rows → {DATA}")
    event_types = ["click", "view", "purchase", "signup", "logout"]
    t0 = time.time()
    with open(DATA, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["user_id", "event_type", "ts", "payload"])
        for i in range(TARGET_ROWS):
            w.writerow([
                random.randint(1, 5_000_000),
                random.choice(event_types),
                1_700_000_000 + i,
                "x" * random.randint(20, 60),
            ])
    print(f"done in {time.time() - t0:.1f}s")

size_gb = os.path.getsize(DATA) / 1024**3
print(f"File on disk: {size_gb:.2f} GB")

In [ ]:
# --- The way every pandas tutorial tells you to do it ----------------------
# pandas loads the ENTIRE file into the Driver's memory. Watch RSS jump — and
# note pandas typically uses 2–3x the file size in RAM. On a big enough file
# this raises MemoryError and kills the Driver. That is the whole point.
print("Driver RSS before load:", round(rss_gb(), 2), "GB")
t0 = time.time()
pdf = pd.read_csv(DATA)
print(f"loaded {len(pdf):,} rows in {time.time() - t0:.1f}s")
print("Driver RSS after load: ", round(rss_gb(), 2), "GB")
print("DataFrame memory:      ", round(pdf.memory_usage(deep=True).sum() / 1024**3, 2), "GB")
print(f"RAM used vs file on disk: ~{rss_gb() / size_gb:.1f}x")

In [ ]:
# --- The Spark way: same file, never loaded onto one machine ---------------
# Spark reads it lazily, in partitions, across the cluster. No Driver blow-up.
sdf = spark.read.csv(f"file:{DATA}", header=True, inferSchema=True)
print("row count via Spark:", sdf.count())   # distributed — Driver stays small
sdf.show(5)

In [ ]:
# More Spark: a distributed aggregation that would strain pandas at this scale.
from pyspark.sql.functions import count
by_type = sdf.groupBy("event_type").agg(count("*").alias("events")).orderBy("events", ascending=False)
by_type.show()

In [ ]:
# Spark reads only what it needs — one column, counted across the cluster.
print("distinct users:", sdf.select("user_id").distinct().count())

In [ ]:
# The same groupBy in pandas needs the WHOLE file in Driver RAM first.
# Spark streamed it in partitions — the Driver never held more than a slice.
sdf.printSchema()

In [ ]:
# Recap:
#   - One machine (the Driver) has fixed memory; pandas loads everything into it.
#   - Big data + single-machine tools = the memory wall (MemoryError / OOM).
#   - Spark spreads the data across many machines, in partitions — no wall.
# Next lecture: how Hadoop first tried to solve this.